# neuro-analog Quick Start Tour

This notebook walks through the core concepts of the neuro-analog framework:

1. Converting a PyTorch model to analog
2. Running mismatch sweeps
3. Understanding energy/latency estimation
4. Interpreting results

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
_ROOT = Path.cwd().parent
sys.path.insert(0, str(_ROOT))

import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from neuro_analog.simulator import (
    analogize,
    mismatch_sweep,
    adc_sweep,
    ablation_sweep,
    count_analog_vs_digital,
)
from neuro_analog.ir.energy_model import HardwareProfile

## 1. Create a Simple Model

We'll use a small MLP for demonstration.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)

model = TinyMLP()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Train on Synthetic Data

In [ ]:
# Generate synthetic data
X, y = make_classification(n_samples=2000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

X_tr = torch.tensor(X_tr, dtype=torch.float32)
X_te = torch.tensor(X_te, dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.long)
y_te = torch.tensor(y_te, dtype=torch.long)

# Train
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(100):
    opt.zero_grad()
    loss = criterion(model(X_tr), y_tr)
    loss.backward()
    opt.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

# Evaluate digital baseline
model.eval()
with torch.no_grad():
    pred = model(X_te).argmax(dim=1)
    digital_acc = (pred == y_te).float().mean().item()
print(f"\nDigital accuracy: {digital_acc*100:.1f}%")

## 3. Convert to Analog

The `analogize()` function replaces PyTorch layers with analog equivalents:

In [ ]:
# Convert to analog with 5% mismatch and 8-bit ADC
analog_model = analogize(model, sigma_mismatch=0.05, n_adc_bits=8)

# Check what was converted
counts = count_analog_vs_digital(analog_model)
print("Analog model composition:")
print(f"  Analog layers:   {counts['analog_layers']}")
print(f"  Analog params:   {counts['analog_params']:,}")
print(f"  Coverage:        {counts['coverage_pct']:.1f}%")
print(f"  Digital layers:  {counts['digital_layers']}")
print(f"  Digital params:  {counts['digital_params']:,}")

## 4. Run Mismatch Sweep

Sweep over different noise levels to see how accuracy degrades.

In [ ]:
def eval_fn(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te).argmax(dim=1)
    return float((pred == y_te).float().mean())

# Create hardware profile for energy/latency estimation
profile = HardwareProfile()

# Run sweep
sweep = mismatch_sweep(
    model,
    eval_fn,
    sigma_values=[0.0, 0.02, 0.05, 0.07, 0.10, 0.12, 0.15],
    n_trials=30,
    n_adc_bits=8,
    hardware_profile=profile,
)

print("\nMismatch sweep results:")
print(f"{'Sigma':>8} {'Accuracy':>10} {'Std':>8} {'Retained':>10}")
for i, sigma in enumerate(sweep.sigma_values):
    print(f"{sigma:8.2f} {sweep.mean[i]*100:9.1f}% {sweep.std[i]*100:7.1f}% {sweep.normalized_mean[i]:9.1%}")

## 5. Energy and Latency Estimates

In [ ]:
print("\nEnergy and latency estimates:")
print(f"  Analog energy:   {sweep.analog_energy_pJ:,.0f} pJ")
print(f"  Digital energy:  {sweep.digital_energy_pJ:,.0f} pJ")
print(f"  Energy saving:    {sweep.energy_saving_vs_digital*100:.1f}%")
print(f"\n  Analog latency:  {sweep.analog_latency_ns:,.0f} ns")
print(f"  Digital latency: {sweep.digital_latency_ns:,.0f} ns")
print(f"  Speedup:          {sweep.speedup_vs_digital:.1f}x")

## 6. Degradation Threshold

The degradation threshold is the noise level where accuracy drops by 10%.

In [ ]:
threshold = sweep.degradation_threshold(max_relative_loss=0.10)
print(f"\n10% degradation threshold: sigma = {threshold:.3f}")
print(f"This means the model tolerates up to {threshold*100:.1f}% conductance mismatch.")

## 7. ADC Precision Sweep

See how ADC bit width affects accuracy at fixed mismatch.

In [ ]:
adc = adc_sweep(
    model,
    eval_fn,
    bit_values=[2, 4, 6, 8, 10, 12],
    n_trials=20,
    sigma_mismatch=0.05,
    hardware_profile=profile,
)

print("\nADC precision sweep results:")
print(f"{'Bits':>6} {'Accuracy':>10} {'Std':>8} {'Retained':>10}")
for i, bits in enumerate(adc.sigma_values):
    print(f"{int(bits):6d} {adc.mean[i]*100:9.1f}% {adc.std[i]*100:7.1f}% {adc.normalized_mean[i]:9.1%}")

## 8. Noise Ablation

Isolate which noise source (mismatch, thermal, quantization) matters most.

In [ ]:
ablation = ablation_sweep(
    model,
    eval_fn,
    sigma_values=[0.0, 0.05, 0.10],
    n_trials=30,
    hardware_profile=profile,
)

print("\nNoise attribution at sigma=0.10:")
for noise_type, result in ablation.items():
    retained = result.normalized_mean[-1]
    print(f"  {noise_type:12s}: {retained:.1%} retained")

## Key Takeaways

1. **Analog conversion is automatic** - `analogize()` handles layer replacement
2. **Energy savings are significant** - analog MACs are ~20x more efficient than digital
3. **Mismatch is the dominant noise source** - thermal noise and quantization have smaller effects
4. **Degradation threshold tells you the tolerance** - higher is better for hardware design

For deeper exploration, see the examples in `examples/` and the unified benchmark in `experiments/unified_benchmark/`.